# A1：Cut-in 加塞场景 —— AEB 算法闭环测试
# A1: Cut-in Scenario — AEB Algorithm Closed-Loop Test

## 实验目标 | Objectives
- 理解 OpenSCENARIO 场景与 esmini 仿真引擎的工作原理
  Understand how OpenSCENARIO scenarios work with the esmini simulation engine
- 掌握 `BaseController` 统一接口（`step(observation) → action`）
  Master the `BaseController` unified interface
- 实现一个基于 TTC（碰撞时间）的紧急制动算法，并通过量化评测
  Implement a TTC-based emergency braking algorithm and pass quantitative evaluation

## 背景知识 | Background
**Cut-in（加塞）**是自动驾驶最危险的场景之一：相邻车道车辆突然变道到自车前方，
自车必须在极短时间内做出制动决策。本实验用 **esmini 真实仿真引擎**驱动目标车辆，
你的 AEB 控制器读取真实仿真状态、输出制动指令，形成真闭环。

**TTC (Time-To-Collision)** = 相对距离 / 相对速度，是 AEB 最核心的决策指标。

## 通过判据 | Pass Criteria
```python
assert not is_collision and min_ttc >= 1.5
```
- 无碰撞，且最小 TTC ≥ 1.5 秒


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
print('✓ labs/common 导入成功')

## 1. 实现你的 AEB 控制器 | Implement Your AEB Controller
继承 `BaseController`，实现 TTC 分级制动逻辑。
Inherit from `BaseController` and implement TTC-based staged braking.

**接口说明 | Interface:**
- 输入 `observation`：`{'ego': 自车状态, 'targets': 目标车列表, 'sim_time': 时间, 'dt': 步长}`
- 输出 `action`：`{'target_a': 目标加速度, 'steer_angle': 转向角}`
- 下面是示例实现，你可以修改阈值和制动策略来优化性能


In [ ]:
class MyAEBController(BaseController):
    """TTC 紧急制动控制器（学生实现区）"""

    def __init__(self, config=None):
        super().__init__(config)
        self.ttc_threshold = self.config.get('ttc_threshold', 3.0)

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}

        # 取最近前车，计算 TTC
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        rel_dist = abs(lead['x'] - ego['x']) - 4.5
        rel_v = ego['v'] - lead['v']

        if rel_v <= 0:
            ttc = float('inf')
        else:
            ttc = rel_dist / rel_v

        # 分级制动（激进版，确保通过）
        if ttc < 3.0:
            a = -5.0  # 强力制动
        else:
            a = 0.0   # 保持速度（不主动加速）

        return {'target_a': a, 'steer_angle': 0.0}

print('✓ 控制器定义完成')

## 2. 运行闭环仿真 | Run Closed-Loop Simulation
启动 **真实 esmini 引擎**（通过 ctypes 调用 `libesminiLib.so`）：
- `SE_InitWithArgs`：加载 OpenSCENARIO 场景文件
- `SE_GetObjectState`：每步读取目标车的真实位置/速度
- `SE_StepDT`：推进仿真 0.05 秒
- 你的控制器每步读取观测、输出动作，形成闭环

> 场景文件由 `gen_cutin.py` 生成：Target 在 t=5s 开始变道，t=8s 完成切入。


In [ ]:
# 0. 生成场景文件（只需要运行一次）
import os
os.chdir('/home/jovyan/labs/ad2026')
!python3 gen_cutin.py
os.chdir('/home/jovyan')
# 检查生成的文件
!ls -lh /home/jovyan/labs/ad2026/*.xosc /home/jovyan/labs/ad2026/*.xodr 2>/dev/null || echo '场景文件生成失败，请检查 gen_cutin.py'


In [ ]:
# 2. 真 esmini 闭环仿真
#
# EsminiEnv 通过 ctypes 调用真实 esmini 库：
# - SE_InitWithArgs 初始化场景
# - SE_ReportObjectPos 上报自车位姿（控制器输出经运动学积分）
# - SE_StepDT 推进一步仿真
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.esmini_env import EsminiEnv

controller = MyAEBController(config={'ttc_threshold': 3.0})
env = EsminiEnv(
    scenario='/home/jovyan/labs/ad2026/cutin.xosc',
    road='/home/jovyan/labs/ad2026/cutin.xodr',
    dt=0.05,
)
obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=14.0)
controller.reset()

trajectory, ttc_history = [], []
min_ttc, is_collision, min_gap = float('inf'), False, float('inf')

for step in range(400):
    action = controller.step(obs)
    obs, done, info = env.step(action)
    t = obs['sim_time']
    ego = obs['ego']
    trajectory.append((t, ego['x'], ego['v'], 0))  # tx 占位
    # 从 targets 计算 TTC（esmini 场景中的真实目标车）
    for tgt in obs.get('targets', []):
        rel_d = abs(tgt['x'] - ego['x']) - 4.5
        min_gap = min(min_gap, rel_d)
        rel_v = ego['v'] - tgt['v']
        if rel_v > 0.1:
            ttc = rel_d / rel_v
            min_ttc = min(min_ttc, ttc)
            ttc_history.append((t, ttc))
        if rel_d < 0.5:
            is_collision = True
    if done or is_collision:
        break

env.close()
print(f'✓ esmini 闭环完成：{len(trajectory)} 步')
print(f'  最小TTC={min_ttc:.2f}s, 最小间隙={min_gap:.1f}m, 碰撞={is_collision}')


## 3. 评测断言 | Evaluation
- `is_collision`：是否发生碰撞（必须为 False）
- `min_ttc`：全程最小 TTC（必须 ≥ 1.5s，越大越安全）


In [ ]:
# 通过判据：无碰撞 且 最小TTC ≥ 1.5s
assert not is_collision, '发生碰撞，测试未通过！'
assert min_ttc >= 1.5, f'最小TTC {min_ttc:.2f}s < 1.5s，测试未通过！'
print('✅ 评测通过！')

## 4. 真实仿真轨迹可视化 | Visualization
下图是**真实仿真数据**绘制的（不是示意动画）：
- 左图：自车速度随时间变化（可看到 AEB 制动介入的时刻）
- 右图：位置-速度轨迹


In [ ]:
import matplotlib.pyplot as plt

times = [t for t, x, v, tx in trajectory]
speeds = [v for t, x, v, tx in trajectory]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(times, speeds)
plt.xlabel('Time (s)'); plt.ylabel('Ego speed (m/s)')
plt.title('AEB Braking Speed Profile')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot([x for t, x, v, tx in trajectory], speeds)
plt.xlabel('Position (m)'); plt.ylabel('Speed (m/s)')
plt.title('Position-Speed Trajectory')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 🏆 挑战任务

1. **调参挑战**：把 `ttc_threshold` 从 2.0 改为 1.5 和 2.5，各跑一次。哪个更安全？哪个更舒适？（提示：看最大减速度）
2. **算法挑战**：实现一个考虑前车加速度的改进版 TTC（提示：`rel_v` 用相对加速度修正），看最小 TTC 能否提升。
3. **场景挑战**：修改 `gen_cutin.py`，让 Target 以 18m/s 切入（比 Ego 快），你的 AEB 还会误触发吗？